# 15 — Variables d'image : PCA, mesures par région, comparaison des représentations (RQ1)

**Question.** Quelle façon de résumer les images aide le plus à prédire la progression ?

| Représentation | Ce que c'est |
|---|---|
| **5 statistiques** (ancien notebook 09) | moyenne, écart-type, percentiles ; images non alignées |
| **Mesures par région** (notebook 09b) | part de liquide / matière grise (IRM) et activité (PET) dans l'hippocampe, l'amygdale, les ventricules, le lobe temporal... |
| **PCA** | on résume chaque volume 3D par ses *k* axes de variation principaux |
| **CNN 3D** (notebook 14) | un réseau de neurones lit le volume entier |

**Règles pour que la comparaison soit honnête**
- même protocole que Nourchene : cible binaire, `StratifiedKFold(5, shuffle=True, random_state=0)`, ROC-AUC en moyenne ± écart-type ;
- la **PCA est calculée dans chaque pli, sur les patients d'apprentissage uniquement** (sinon l'information du pli de test
  fuiterait dans les variables) ;
- le nombre de composantes *k* n'est **pas choisi sur le test** : on rapporte k = 10 et k = 20 ;
- les variables du tableau sont mesurées à la date de l'image (notebook 07b).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"..\configs")
sys.path.append(r"..\src")
from config_projet import C
import alignement_mni as A

EXCLURE_DEJA_DEMENTS = True   # meme choix que les notebooks 10 et 14
KS = [10, 20]                 # nombres de composantes de la PCA (non choisis sur le test)

prep = sorted(C.COHORTE.glob("cible_ancree_image_*"))[-1]       # notebook 07b
feat = sorted(C.COHORTE.glob("features_image_mni_*"))[-1]       # notebook 09b
ancien = sorted(C.COHORTE.glob("features_image_[0-9]*"))[-1]    # notebook 09 (5 statistiques)
print("Tabulaire :", prep.name, "| images alignees :", feat.name, "| anciennes stats :", ancien.name)

jeu = pd.read_csv(prep / "jeu_modelisation_112_ancre.csv")
reg = pd.read_csv(feat / "features_regions.csv")
old = pd.read_csv(ancien / "features_image.csv")
for t in (jeu, reg, old):
    t["PTID"] = t["PTID"].astype(str).str.strip().str.upper()
old = old.rename(columns={c: "ancien_" + c for c in old.columns if c != "PTID"})

donnees = (jeu.merge(reg, on="PTID", validate="one_to_one")
              .merge(old, on="PTID", validate="one_to_one")
              .dropna(subset=["progression_image"]).reset_index(drop=True))
if EXCLURE_DEJA_DEMENTS:
    n0 = len(donnees)
    donnees = donnees.loc[donnees["deja_dement"] == 0].reset_index(drop=True)
    print("Patients deja dements exclus :", n0 - len(donnees))
y = donnees["progression_image"].astype(int).to_numpy()
print("Participants :", len(y), "| progressent :", int(y.sum()), f"({y.mean() * 100:.1f}%)")

TAB = [c for c in ["sexe", "education", "age_reference", "APOE_e4", "CDR_global", "CDR_SB", "MMSE_debut",
                   "ADAS13", "FAQ_debut", "ABETA42", "TAU", "PTAU", "dx_image"] if donnees[c].notna().sum() > 0]
REG_IRM = [c for c in reg.columns if c.startswith("IRM_") and (c.endswith("_LCR") or c.endswith("_MG"))]
REG_PET = [c for c in reg.columns if c.startswith("PET_") and c not in ("PET_corr_matiere_grise", "PET_essais")]
OLD_IRM = [c for c in old.columns if c.startswith("ancien_IRM_")]
OLD_PET = [c for c in old.columns if c.startswith("ancien_PET_")]
print("Variables : tabulaire", len(TAB), "| regions IRM", len(REG_IRM), "| regions PET", len(REG_PET),
      "| anciennes stats", len(OLD_IRM) + len(OLD_PET))

## 1. Les volumes alignés, réduits pour la PCA

Chaque volume 64³ est réduit à 32³ (moyenne par blocs de 2×2×2) et on ne garde que les voxels **à l'intérieur du
cerveau MNI** (masque du modèle, identique pour tous les patients et indépendant des données).

In [ ]:
modele = A.ModeleMNI(C.DATA / "08_Modele_MNI")
masque64 = A._vers_cube(modele, modele.masque.astype(np.float32)) > 0.5
masque32 = masque64.reshape(32, 2, 32, 2, 32, 2).mean(axis=(1, 3, 5)) > 0.5
print("Voxels dans le cerveau (32^3) :", int(masque32.sum()))


def volumes(modalite):
    piles = []
    for p in donnees["PTID"]:
        v = np.load(feat / "volumes_mni" / f"{p}_{modalite}.npy")
        v = v.reshape(32, 2, 32, 2, 32, 2).mean(axis=(1, 3, 5))
        piles.append(v[masque32])
    return np.stack(piles).astype(np.float32)


VOL_IRM, VOL_PET = volumes("IRM"), volumes("PET")
print("IRM :", VOL_IRM.shape, "| PET :", VOL_PET.shape)

## 2. Évaluation : mêmes plis pour toutes les représentations

Chaque représentation est une fonction qui, pour un pli (indices d'apprentissage / de test), renvoie les variables.
Deux classifieurs : une **random forest** (celui du notebook 10) et une **régression logistique** (mieux adaptée
quand il y a beaucoup de variables pour peu de patients).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, accuracy_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def bloc_colonnes(cols):
    def f(app, test):
        X = donnees[cols].to_numpy(dtype=float)
        return X[app], X[test]
    return f


def bloc_pca(vol, k):
    def f(app, test):
        pca = PCA(n_components=k, svd_solver="randomized", random_state=0).fit(vol[app])  # pli d'apprentissage seulement
        return pca.transform(vol[app]), pca.transform(vol[test])
    return f


def assembler(*blocs):
    def f(app, test):
        paires = [b(app, test) for b in blocs]
        return np.hstack([p[0] for p in paires]), np.hstack([p[1] for p in paires])
    return f


def classifieurs():
    return {
        "Random forest": Pipeline([("imp", SimpleImputer(strategy="median")),
                                   ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0))]),
        "Regression logistique": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                           ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    }


def evaluer(fabrique):
    res = {nom: {"roc_auc": [], "balanced_accuracy": [], "f1": [], "accuracy": []} for nom in classifieurs()}
    for app, test in CV.split(np.zeros(len(y)), y):
        Xa, Xt = fabrique(app, test)
        for nom, modele_clf in classifieurs().items():
            modele_clf.fit(Xa, y[app])
            proba = modele_clf.predict_proba(Xt)[:, 1]
            pred = (proba >= 0.5).astype(int)
            res[nom]["roc_auc"].append(roc_auc_score(y[test], proba))
            res[nom]["balanced_accuracy"].append(balanced_accuracy_score(y[test], pred))
            res[nom]["f1"].append(f1_score(y[test], pred, zero_division=0))
            res[nom]["accuracy"].append(accuracy_score(y[test], pred))
    return res

In [ ]:
REPRESENTATIONS = {
    "Tabulaire seul": bloc_colonnes(TAB),
    "5 statistiques d'image (ancien 09, non alignees)": bloc_colonnes(OLD_IRM + OLD_PET),
    "Regions IRM": bloc_colonnes(REG_IRM),
    "Regions PET": bloc_colonnes(REG_PET),
    "Regions IRM + PET": bloc_colonnes(REG_IRM + REG_PET),
}
for k in KS:
    REPRESENTATIONS[f"PCA IRM (k={k})"] = bloc_pca(VOL_IRM, k)
    REPRESENTATIONS[f"PCA PET (k={k})"] = bloc_pca(VOL_PET, k)
    REPRESENTATIONS[f"PCA IRM + PET (k={k} chacune)"] = assembler(bloc_pca(VOL_IRM, k), bloc_pca(VOL_PET, k))
REPRESENTATIONS["Tabulaire + 5 statistiques (ancien 09)"] = assembler(
    bloc_colonnes(TAB), bloc_colonnes(OLD_IRM + OLD_PET))
REPRESENTATIONS["Tabulaire + regions IRM + PET"] = assembler(bloc_colonnes(TAB), bloc_colonnes(REG_IRM + REG_PET))
for k in KS:
    REPRESENTATIONS[f"Tabulaire + PCA IRM + PET (k={k})"] = assembler(
        bloc_colonnes(TAB), bloc_pca(VOL_IRM, k), bloc_pca(VOL_PET, k))

RESULTATS = {}
for nom, fabrique in REPRESENTATIONS.items():
    RESULTATS[nom] = evaluer(fabrique)
    print(f"{nom:55s} RF {np.mean(RESULTATS[nom]['Random forest']['roc_auc']):.3f} | "
          f"LR {np.mean(RESULTATS[nom]['Regression logistique']['roc_auc']):.3f}", flush=True)

## 3. Tableau de résultats (ROC-AUC, moyenne ± écart-type sur 5 plis)

In [ ]:
def ms(v):
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"


lignes = []
for nom, r in RESULTATS.items():
    lignes.append({
        "Representation": nom,
        "AUC random forest": ms(r["Random forest"]["roc_auc"]),
        "AUC regression logistique": ms(r["Regression logistique"]["roc_auc"]),
        "_rf": np.mean(r["Random forest"]["roc_auc"]),
        "_lr": np.mean(r["Regression logistique"]["roc_auc"]),
    })
tableau = pd.DataFrame(lignes)
tableau["_max"] = tableau[["_rf", "_lr"]].max(axis=1)
tableau = tableau.sort_values("_max", ascending=False)
display(tableau[["Representation", "AUC random forest", "AUC regression logistique"]].reset_index(drop=True))

tab_rf = np.mean(RESULTATS["Tabulaire seul"]["Random forest"]["roc_auc"])
tab_lr = np.mean(RESULTATS["Tabulaire seul"]["Regression logistique"]["roc_auc"])
print(f"Reference : tabulaire seul = {tab_rf:.3f} (RF) / {tab_lr:.3f} (LR)")
print("Avec", len(y), "patients, un ecart de moins de ~0,05 d'AUC n'est pas distinguable du bruit "
      "(ecart-type entre plis : 0,05 a 0,15).")

SORTIE = C.sortie("representations_image")
tableau.drop(columns=["_rf", "_lr", "_max"]).to_csv(SORTIE / "RQ1_representations.csv", index=False, encoding="utf-8-sig")
detail = [{"representation": n, "classifieur": c, "metrique": m, "moyenne": np.mean(v), "ecart_type": np.std(v)}
          for n, r in RESULTATS.items() for c, d in r.items() for m, v in d.items()]
pd.DataFrame(detail).to_csv(SORTIE / "RQ1_representations_detail.csv", index=False, encoding="utf-8-sig")
print("Enregistre dans :", SORTIE)

In [ ]:
import matplotlib.pyplot as plt

ordre = tableau.sort_values("_max")["Representation"].tolist()
fig, ax = plt.subplots(figsize=(9, 0.42 * len(ordre) + 1.5), constrained_layout=True)
pos = np.arange(len(ordre))
for decalage, (clf, couleur) in zip([-0.2, 0.2], [("Random forest", "#4C72B0"), ("Regression logistique", "#DD8452")]):
    moy = [np.mean(RESULTATS[n][clf]["roc_auc"]) for n in ordre]
    ect = [np.std(RESULTATS[n][clf]["roc_auc"]) for n in ordre]
    ax.barh(pos + decalage, moy, height=0.38, xerr=ect, color=couleur, capsize=2, label=clf)
ax.axvline(0.5, color="grey", ls="--", lw=1)
ax.set_yticks(pos)
ax.set_yticklabels(ordre, fontsize=8)
ax.set_xlim(0.3, 1.0)
ax.set_xlabel("ROC-AUC (validation croisee 5 plis)")
ax.set_title("Quelle representation des images aide a predire la progression ?")
ax.legend(loc="lower right", fontsize=8)
fig.savefig(SORTIE / "RQ1_representations.png", dpi=130)
plt.show()

## 4. Que capture la PCA ? (part de variance expliquée)

Si les premières composantes expliquent beaucoup de variance, elles décrivent surtout des différences *entre
tous les cerveaux* (taille, forme, contraste), pas forcément la maladie.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
for a, (nom, vol) in zip(ax, [("IRM", VOL_IRM), ("PET", VOL_PET)]):
    pca = PCA(n_components=20, svd_solver="randomized", random_state=0).fit(vol)
    a.plot(np.arange(1, 21), np.cumsum(pca.explained_variance_ratio_) * 100, marker="o")
    a.set_title(f"PCA {nom} : variance expliquee cumulee")
    a.set_xlabel("composantes"); a.set_ylabel("%")
plt.show()

## 5. Avant / après correction et CNN

Les tableaux ci-dessous lisent les résultats déjà enregistrés par les notebooks 10 et 14 (versions *initiales*, puis
*corrigées*). Ils s'affichent seulement si ces notebooks ont été exécutés.

In [ ]:
def dernier(motif):
    d = sorted(C.COHORTE.glob(motif))
    return d[-1] if d else None


for titre, motif, fichier in [
    ("Notebook 10 - donnees INITIALES (cible decalee, 5 statistiques)", "fusion_RQ1_RQ3_[0-9]*", "RQ1_comparaison.csv"),
    ("Notebook 10 - donnees CORRIGEES (cible ancree, regions du cerveau)", "fusion_RQ1_RQ3_corrigee_*", "RQ1_comparaison.csv"),
    ("Notebook 14 - CNN 3D, donnees INITIALES", "cnn_3d_[0-9]*", "resultats_CNN.csv"),
    ("Notebook 14 - CNN 3D, donnees CORRIGEES", "cnn_3d_corrigee_*", "resultats_CNN.csv"),
]:
    d = dernier(motif)
    print("\n" + titre, "->", d.name if d else "pas encore execute")
    if d is not None and (d / fichier).exists():
        display(pd.read_csv(d / fichier, index_col=0))